Conectando com o assunto anterior (Spark e Big Data), a escolha do **tipo (ou formato) de arquivo** em que você armazena seus dados é tão crucial quanto a definição dos tipos de dados em si. Ela dita a velocidade das suas queries, o espaço de armazenamento que você vai gastar e o custo da sua infraestrutura.

No ecossistema de Big Data, os formatos de arquivo são divididos principalmente pela forma como organizam os dados no disco: **Orientados a Linhas** ou **Orientados a Colunas**.

---

## Os Principais Formatos de Arquivo em Big Data

### 1. Apache Parquet (Colunar)

É o formato padrão e o "queridinho" do Apache Spark. Em vez de salvar os dados linha por linha, ele agrupa os dados por colunas.

* **Vantagens:** Se você tem uma tabela com 100 colunas, mas sua query só usa 3, o Spark lê *apenas* os dados dessas 3 colunas (**Column Pruning**). Além disso, dados do mesmo tipo juntos comprimem muito melhor (redução de até 75% no espaço em disco).
* **Ideal para:** Cenários de Analytics, Data Warehouses, Lakehouses e queries complexas de leitura (OLAP).

### 2. Apache Avro (Baseado em Linhas)

O Avro armazena os dados no formato de linhas, guardando o esquema (*schema*) em um cabeçalho JSON compacto junto com os dados binários.

* **Vantagens:** Excelente para operações de escrita rápida, pois adicionar uma nova linha é muito simples. Ele brilha na **evolução de schema** (quando sua tabela ganha ou perde colunas ao longo do tempo sem quebrar o sistema).
* **Ideal para:** Sistemas de mensageria e streaming em tempo real (como Apache Kafka) e arquiteturas de ingestão de dados (onde a escrita precisa ser imediata).

### 3. Apache ORC (Colunar)

Muito semelhante ao Parquet, o *Optimized Row Columnar* (ORC) foi criado originalmente para o ecossistema Apache Hive, mas é totalmente suportado pelo Spark.

* **Vantagens:** Oferece taxas de compressão altíssimas e possui índices integrados (como valores mínimos e máximos por bloco), acelerando muito a filtragem de dados.
* **Ideal para:** Ambientes que utilizam muito o Hive integrados ao Spark.

### 4. JSON (Semi-estruturado)

Formato de texto amplamente utilizado na web. O Spark consegue ler arquivos JSON e inferir o esquema automaticamente.

* **Vantagens:** Altamente legível por humanos e flexível para dados que mudam de estrutura constantemente.
* **Desvantagens:** É muito pesado. Não possui compressão nativa e o Spark precisa ler o arquivo inteiro para entender a estrutura, o que o torna lento para Big Data.

### 5. CSV / TSV (Texto Plano)

O formato mais tradicional e comum no dia a dia.

* **Vantagens:** Universal. Qualquer ferramenta, do Excel ao Spark, lê e escreve em CSV.
* **Desvantagens:** Não guarda metadados (tipos de dados), não suporta aninhamento (como arrays ou mapas) e a leitura é sequencial e lenta em larga escala.

---

## Tabela Comparativa de Formatos

| Formato | Organização | Armazenamento do Schema | Eficiência de Leitura | Eficiência de Escrita |
| --- | --- | --- | --- | --- |
| **Parquet** | Colunar | Embutido no arquivo | ✨ Excelente | ⚠️ Moderada |
| **ORC** | Colunar | Embutido no arquivo | ✨ Excelente | ⚠️ Moderada |
| **Avro** | Linhas | Embutido (JSON) | ⚠️ Moderada | ✨ Excelente |
| **JSON** | Textual | Implícito / Texto | ❌ Ruim | ⚠️ Moderada |
| **CSV** | Textual | Não possui (só Header) | ❌ Ruim | ✨ Excelente |

---

> 💡 **Dica de Ouro:** Se você está desenhando um Data Lake moderno, a tendência atual é envelopar arquivos Parquet usando camadas de gerenciamento como **Delta Lake** ou **Apache Iceberg**. Eles adicionam recursos como transações ACID (garantia de que a escrita deu certo ou errado por completo) e viagem no tempo (consultar como os dados estavam no passado).

In [0]:
data  = (
    spark
        .read
        .format('csv')
        .option('header', 'true')
        .option('inferSchema', 'true')
        .load('/Volumes/learn_databricks/schema/volume/Clientes/Clientes.csv')
)

In [0]:
data.limit(3).display()

id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals
0,2017-11-01T14:45:41.000Z,Marta,Jesus,null,9 9102-7834,Brasil,Acre,null,null,Conjunto 16
1,2017-10-16T00:50:39.000Z,Luana,Almeida,null,9 7328-8718,Brasil,Rio Grande do Sul,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
2,2018-06-16T17:51:29.000Z,Frida,Mendes,frida@meu_email.com,9 5906-7552,Brasil,São Paulo,Avenida 59 do Estado São Paulo,534.0,null


Salvando parquet

In [0]:
data\
    .write\
    .format("parquet")\
    .options(**{"compression": "snappy"})\
    .save("/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet")

In [0]:
data\
    .write\
    .format("json")\
    .save("/Volumes/learn_databricks/schema/volume/Clientes/json/Clientes.json")

Para decidir qual formato de arquivo usar no seu pipeline de dados, a regra de ouro é: **olhe para o padrão de acesso aos dados**. Sua aplicação vai escrever muito e ler pouco? Vai ler colunas específicas para gerar relatórios? Precisa trocar dados com sistemas externos?

Aqui está o guia prático de quando usar cada um:

---

### 1. Escolha **Parquet** quando:

* **O foco for Analytics (OLAP):** Você precisa rodar queries SQL agregando milhões de linhas, mas usando poucas colunas (ex: calcular a média de faturamento por região).
* **Armazenamento for um custo crítico:** O Parquet reduz drasticamente o tamanho dos arquivos em disco graças à compressão por coluna.
* **O Spark for o motor principal:** É o formato nativo e mais otimizado para o ecossistema Spark.

### 2. Escolha **Avro** quando:

* **O foco for Ingestão e Streaming:** Você está capturando dados em tempo real (ex: com Apache Kafka) onde o sistema precisa escrever linha por linha muito rápido.
* **O Schema muda constantemente:** Se a sua aplicação ganha colunas novas toda semana, o Avro gerencia essa transição de forma transparente através da "evolução de schema" (evitando que suas tabelas quebrem).
* **Você precisa ler a linha inteira:** Consultas que exigem o registro completo (todas as colunas de uma vez).

### 3. Escolha **CSV / TSV** quando:

* **Interoperabilidade for prioridade:** Você precisa exportar dados para que um analista de negócios abra no Excel ou para importar em uma ferramenta legada que não aceita Parquet.
* **Cargas pequenas e rápidas:** Para arquivos de configuração ou tabelas de dicionário muito pequenas (poucos megabytes).

### 4. Escolha **JSON** quando:

* **Dados semi-estruturados vindos da Web:** Coleta de dados de APIs REST, logs de servidores web ou dados de sensores IoT onde a estrutura pode variar de um registro para outro.
* **Landing Zone (Camada Raw):** Usado apenas como a primeira parada do dado bruto no Data Lake, antes de ser limpo e convertido para Parquet.

### 5. Escolha **ORC** quando:

* **Seu ambiente for muito focado em Apache Hive:** O ORC oferece vantagens de indexação e compressão ligeiramente superiores ao Parquet quando integrado diretamente com o ecossistema Hive. Se o seu foco exclusivo for Spark, prefira Parquet.

---

## O Resumo da Ópera (Regra de Bolso)

> 🟢 **Para a camada de consumo (BI/Relatórios):** Use **Parquet** (ou Delta/Iceberg).
> 🟡 **Para a camada de captura (Streaming/Ingestão):** Use **Avro**.
> 🔴 **Para trocar dados com humanos ou sistemas externos:** Use **CSV** ou **JSON**.

.